# 学习笔记
> 更新说明:
> 本项目与一轮提交的相比,做出了以下调整:
> 1. 将数据处理分为dataset和dataloader(分区更有界限些)
> 2. 加入了config.py用以记录参数,路径以及各类列表
> 3. 将predict重分为了evaluate,predict,分别用以画图保存和预测新数据
> 4. 在1~3的基础上对每个板块进行了调整,让项目结构更清晰
> 5. 加入了L2正则方法
> 6. 多采用了ROC图像和混淆矩阵对模型进行评估
> 7. 加入对GPU的选择使用
> 8. 改善精确度和损失
> 9. 处理了Name特征,增加了title的特征
> 10. 提交了多篇各板块学习笔记




## 专题一：提交内容
### 1.数据清洗和特征处理方法；

本项目采用了删除特征，根据特点填补缺失值，无量纲化的标准化，创造新特征的方法。具体描述如下：

- 删除特征：训练集中我删除了"PassengerId"（无效）,  "Survived"（单独处理labal以防止数据泄露）, "Ticket"（编号可能隐藏一些信息，比如编号小的可能地位高实力强，但偶然性太大，故舍去）, "Cabin"（客舱号缺失太多，且因为人员活动，如果补全并采用可能对模型造成误导，误差大）

- 根据特点填补缺失值：数值型采用中位数填补，如Age,Fare.(不用均值是因为均值受异常值的影响较大)。类别型采用众数填补（有多个众数时选择第一个）

- 无量纲化的标准化：一般对处理后的数据范围没有太严格的要求都选择标准化，而非归一化（标准化受极端异常值的影响较小）（这里有个疑惑，我尝试了筛选标准化之后的数据，显示有超过了【-10，10】的区间的，但打印出来的部分序列数值都很小（没有异常数据））

- 创造新特征：利用SibSp，Parch合并成为了新的特征FamilySize，IsAlone.这些数据可能反馈出，乘客可能因等待生存几率下降，或因互助让生存几率上升。（有了FamilySize，还要创造IsAlone是因为，模型不能清晰认识到独行的影响，FamilySize的1、2、3、4对模型来说是等差的，但其实1和2有着天差地别的差距，所以创造了IsAlone判断是否独行）;也用了Name创造出来了title特征.

- 补充：本题没有异常数据，如果有，可采用如下方法，将异常值归为NaN，再和缺失值一起处理。（如果要删异常值，应该要记录一下哪些数据被删了）



In [ ]:
bad = (data['age'] < 0) | (data['age'] > 120)
data.loc[bad, 'age'] = np.nan

### 2.模型结构、损失函数、优化器及主要超参数

- 模型结构：利用nn.Module,构造了一个深度学习的网络，采用了线性函数和激活函数结合，利用Dropout进行正则化（优化时加入了L2正则化，还可以把一些删去的特征加上，并用L1正则化进行间接选择），一共有两个隐藏层（在设置模型时是没有设置输入层的）。 实操时模型采用了先预测，再算Loss，再反向传播、结合Adamw改参数（记得清除之前的梯度），训练完后关梯度，算测试集准确率

- 损失函数：是利用torch中的nn.CrossEntropyLoss（交叉熵损失），适合分类任务，可以在[PyTorch 官方 CrossEntropyLoss 文档] https://pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html 中查看细节（太多了，记不住）

- 优化器：运用Adamw优化器（Adam 优化器是一个能为每个参数调整学习率，并且可以根据梯度进行动态调整的系数优化器。他结合了Momentum和RMSProp的优点,但是Adamw会影响我加入的L2正则

- 主要超参数

| 超参数                 | 取值    |
|------------------------|---------|
| 隐藏层神经元数 h1 / h2 | 32 / 16 |
| Dropout 率             | 0.2     |
| 学习率 lr              | 0.01    |
| 训练轮数 epochs        | 200     |
| 测试集比例             | 0.2     |
| 随机种子               | 42      |
| 优化器                 | Adamw   |
| WEIGHT_DECAY           | 0.5     |

### 3.数据划分方式、随机种子和实验配置

- 数据划分方式:先将幸存和没幸存分开，打乱后，按训练集、测试集比例将幸存和没幸存划分，再相应地拼装在一起，最后再打乱一次。

- 随机种子：设定了固定种子——42，便于调试，消除部分偶然性

- 实验配置：
 1. 运行环境

| 项 | 值                         |
|---|----------------------------|
| 操作系统 | Windows 11                 |
| 解释器 | pycharm                    |
| Python | 3.14.7                     |
| PyTorch | 2.11.0+cu128               |
| NumPy / pandas | 2.5.2 / 3.0.5              |
| matplotlib | 3.11.1                     |
| 实际计算设备 | **GPU**(也可用CPU进行训练) |

 2. 产出文件

| 文件                          | 说明       |
|-------------------------------|------------|
| `model/titanic.pth`           | 模型权重   |
| `loss_curve.png`              | 损失曲线   |
| `ROC_curve.png`               | ROC曲线    |
| `accuracy_curve.png`          | 准确率曲线 |
| `titanic学习笔记.ipynb` 等5个 | 学习笔记   |


 3. 实验结果

| 指标                         | 数值                    |
|------------------------------|-------------------------|
| 首轮 loss                    | 0.6451                  |
| 末轮 loss                    | 0.3948                  |
| 最终测试准确率（第 200 轮）  | **0.8531**              |
| 全程最高测试准确率           | **0.8814**              |
| 盲猜遇难基线                 | 0.6158                  |
| 预测示例（一等舱 25 岁女性） | 预测幸存，概率为 0.9267 |
| AUC                      |0.9079195898542904   |


### 4.训练过程中遇到的问题及解决方法
内容有些多，单独列了一个专题（见专题2）
### 5.测试集准确率、实验结果分析和心得体会
- 测试集准确率：可以达到85.31%
- 实验结果分析:损失有较明显的下降，精确度显著增加（显著高于盲猜）,测试集与训练集acc相近,未出现欠拟合与过拟合。AUC约等于0.9,接近1,模型召回率较高,误判率较低
- 心得体会：心得体会不多，但也单列了一个专题（见专题3）

## 专题二：训练过程中遇到的问题及解决方法
> 说明1：部分由ai生成，再由本人修改

> 说明2：这里只是部分，更多在源代码中有注释

| 问题                           | 答案                                                                                                                                                        |
|--------------------------------|-------------------------------------------------------------------------------------------------------------------------------------------------------------|
| `df: {__getitem__, copy}`      | 不是可执行代码，是"能力清单"：能 `df['列']` 取值、能 `.copy()`                                                                                              |
| torch.tensor是什么             | 相当于 np.array，但能上 GPU、能自动求梯度；是"预处理结果 → 网络"的桥                                                                                        |
| dtype vs astype 区别           | dtype 是**属性**（查现在啥类型）；astype 是**方法**（返回改类型后的新数组）                                                                 |
| `np.where(...)[0]` 的 `[0]`    | np.where 返回"元组包数组"，[0] 把真正的下标数组剥出来，才能 `[:n]` 切片                                                                                     |
| FEATURE_ORDER 为何能固定顺序   | list 有序 + `feats[ORDER]` 强制重排。网络只认"第几个输入口"，错位会**安静地算错**                                                                           |
| 函数里的变量怎么搬出去         | 局部变量函数结束即销毁。正规做法：**一律用 return 传**（可返多个值/字典/对象）|
| 环境/脚本定位                  | `os.path.dirname(os.path.abspath(__file__))` 拿脚本所在目录                                          |


#### 2. 报错表（学长，放下屠刀，这个是ai根据我问他的问题生成的（所以也不太全），我修改成了我能看懂的样子，但我背不到啊~~~）

| 报错 | 原因                                       | 改法                                                                                |
|---|--------------------------------------------|-------------------------------------------------------------------------------------|
| `KeyError: ['PassengerId'] not found` | 同一列被 drop 两次                         | 只留一个 drop                                                                       |
| `KeyError: ['FamilySize'] not in index` | FamilySize 是新创造的特征，却在创造前调用  | 改顺序                                                                              |
| `AttributeError: 'tuple' object has no attribute 'drop'` | `return self, df` 打成元组，破坏赋值链     | 删掉 self                                                                           |
| `TypeError: int() ... not DataFrame` | `a.split(df)` 多传的 df 被当 test_size     | split 再接收一个 df 形参                                                            |
| `TypeError: module() takes at most 2 arguments (3 given)` | `class Net(nn)` —— nn 是模块不能继承       | `class Net(nn.Module)`                                                              |
| `TypeError: empty(): argument 'size' failed...` | `Net(nn.Module)` 把类当维数传进去了        | 传整数 `Net(ncol)`                                                                  |
| `KeyError: ['FamilySize','Sex_male'...] not in index` | 用原始 df 取新的特征                       | `X = df2[ORDER].values.astype('float32')`                                           |
| `IndexError: Target 1 is out of bounds` | CrossEntropyLoss 要 2，给的 1              | 模型输出改为2                                                                       |
| `Cannot assign modules before Module.__init__()` | 漏了 `super().__init__()`                  | `__init__` 第一行补齐                                                               |
| `ImportError: cannot import name 'draw'` | 函数的引用成环了                           | 改成单向，main控制全部                                                              |
| `mat1 and mat2 shapes cannot be multiplied (5x12 and 13x32)` | 喂了 12 维                                 | 特征按 ORDER 补齐 13 维                                                             |
| `all the input array dimensions must match` | concatenate 时拼不上（维度问题）           | 竖接看列数、横接看行数                                                              |
| `'numpy.ndarray' object is not callable` | `.values()` 多加了括号                     | `.values`  无括号                                                    |
| `OMP: Error #15` | torch 与 numpy 各带一份 OpenMP | `main.py` 首行（`import torch` 之前）设 `os.environ['KMP_DUPLICATE_LIB_OK']='TRUE'` |



## 专题三：心得体会
二轮再次完善titanic这个项目,有了更多的收获,对于这种模型构建的流程更清晰,也学会了更多的方法,也将一轮留下的优化方向完成了一些,虽然还是有很多要完善的地方(比如学长说的调度器,我对它还有些二懵二懵的,就暂时没加入代码里,暂且作为优化方向继续了解).以下是一些解决了的问题:
1. 什么时候我需要把函数打包成类的形式？什么时候不需要？

| * | 函数 | 类 |
|---|---|---|
| 上次的数据 |  调完就忘 |  存在 `self` 里 |
| 代码量 | 少、直接 | 多一个 `__init__` 样板 |
| 难度 | 低 | 高（self、实例化、方法） |
| 适合 | 算一下就完事 | 反复用、要复用 |
| 典型 | `len()` | `nn.Module` |

2. np.where()用法
- np.where 有三参数版本：np.where(条件, A, B) = 满足条件取 A，否则取 B，相当于三元表达式。
- np.where 有另一个用途e.g.surv = np.where(df_survived == 1)[0].先返回哪些位置是Ture,返回的是一个元组，如果是二维的则一个元组记录行坐标，一个元组记录纵坐标，一一对应才有点坐标(至于二维的如何取出为列表，还在研究中)
3. 为什么必须有dataset
- DataLoader 有个硬规矩：只接收 Dataset 对象，不然报错
4. predict里面是需要平均数和方差这两个训练集在main中算出来的数据的。但是我又不能直接从main中去把这两个数据import过来，不然会造成import的循环，那我怎么在predict之中用这两个数据呢?
- 把predict的函数的形参加入mean,std，在main中多输入两个实参，不直接把mean,std搬到predict来

## 专题四：优化方向

1. 从Name特征中提取出名字长度，同名人数等弱相关特征，用L1正则化间接选择

2. 视情况加入早停(早停需要验证集的加入,而题中明确说了分为训练集和测试集,故而暂时没加早停)

3. 直接用已有函数进行独热编码

4. 加入调度器提高模型的表现

5. 敬请期待!